In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score
from rdkit import Chem
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

DATA = Path("data") if Path("data").exists() else Path("../data")
PLOTS = Path("results/plots") if Path("results").exists() else Path("../results/plots")

In [2]:
X = np.load(DATA / "egfr_fingerprints.npy")
df = pd.read_csv(DATA / "egfr_features_and_labels.csv")
y = df['label'].values

train_idx = np.load(DATA / "scaffold_train_idx.npy")
test_idx = np.load(DATA / "scaffold_test_idx.npy")
X_train, y_train = X[train_idx], y[train_idx]
X_test, y_test = X[test_idx], y[test_idx]
print("Train:", X_train.shape, "| Test:", X_test.shape)

# Retrain Random Forest on the scaffold training set 
rf = RandomForestClassifier(n_estimators=500, class_weight='balanced', random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
auc = roc_auc_score(y_test, rf.predict_proba(X_test)[:, 1])
print(f"Scaffold-split ROC-AUC: {auc:.3f} (notebook 3 reported 0.893)")

Train: (9939, 2048) | Test: (2485, 2048)
Scaffold-split ROC-AUC: 0.893 (notebook 3 reported 0.893)


In [3]:
from sklearn.metrics import roc_auc_score, average_precision_score

# Scores for every test molecule (models trained on the scaffold training set only)
rf_scores = rf.predict_proba(X_test)[:, 1]

def tanimoto_matrix(A, B):
    A = A.astype(np.float32)
    B = B.astype(np.float32)
    inter = A @ B.T
    union = A.sum(axis=1)[:, None] + B.sum(axis=1)[None, :] - inter
    return inter / np.maximum(union, 1)

S = tanimoto_matrix(X_test, X_train)
nn_scores = S[:, y_train == 1].max(axis=1) - S[:, y_train == 0].max(axis=1)
del S

act_idx = np.where(y_test == 1)[0]
inact_idx = np.where(y_test == 0)[0]
PREVALENCE = 0.05
n_act = int(round(PREVALENCE * len(inact_idx) / (1 - PREVALENCE)))
N = n_act + len(inact_idx)
print(f"Library: {N} molecules = {n_act} actives + {len(inact_idx)} measured inactives ({n_act / N:.1%} actives)")
print(f"Random baseline: EF = 1.0, Precision = PR-AUC = {n_act / N:.3f} | maximum possible EF at 1%: {N / n_act:.1f}")

def screen_metrics(scores, labels, fracs=(0.01, 0.05)):
    ranked = labels[np.argsort(-scores)]
    n_lib, n_hit = len(labels), labels.sum()
    out = {"ROC-AUC": roc_auc_score(labels, scores),
           "PR-AUC": average_precision_score(labels, scores)}
    for f in fracs:
        k = int(np.ceil(f * n_lib))
        hits = ranked[:k].sum()
        out[f"EF {int(f * 100)}%"] = (hits / k) / (n_hit / n_lib)
        out[f"Precision {int(f * 100)}%"] = hits / k
    return out

rng = np.random.default_rng(42)
results = {"Random Forest": [], "Tanimoto 1-NN": []}
for _ in range(100):
    chosen = rng.choice(act_idx, size=n_act, replace=False)
    lib = np.concatenate([chosen, inact_idx])
    labels = y_test[lib]
    for name, sc in [("Random Forest", rf_scores), ("Tanimoto 1-NN", nn_scores)]:
        s = sc[lib] + 1e-9 * rng.random(len(lib))  # random tie-breaking
        results[name].append(screen_metrics(s, labels))

rows = []
for name, res in results.items():
    d = pd.DataFrame(res)
    for m in d.columns:
        rows.append({"Metric": m, "Model": name, "Mean": d[m].mean(),
                     "5th pct": d[m].quantile(0.05), "95th pct": d[m].quantile(0.95)})
summary = pd.DataFrame(rows).round(3).sort_values(["Metric", "Model"])
print(summary.to_string(index=False))

Library: 1049 molecules = 52 actives + 997 measured inactives (5.0% actives)
Random baseline: EF = 1.0, Precision = PR-AUC = 0.050 | maximum possible EF at 1%: 20.2
      Metric         Model   Mean  5th pct  95th pct
       EF 1% Random Forest 16.450   12.837    20.173
       EF 1% Tanimoto 1-NN  6.730    3.668     9.170
       EF 5% Random Forest  9.234    7.612    10.677
       EF 5% Tanimoto 1-NN  8.275    7.213     9.535
      PR-AUC Random Forest  0.505    0.404     0.596
      PR-AUC Tanimoto 1-NN  0.315    0.248     0.370
Precision 1% Random Forest  0.815    0.636     1.000
Precision 1% Tanimoto 1-NN  0.334    0.182     0.455
Precision 5% Random Forest  0.458    0.377     0.529
Precision 5% Tanimoto 1-NN  0.410    0.358     0.473
     ROC-AUC Random Forest  0.895    0.857     0.931
     ROC-AUC Tanimoto 1-NN  0.875    0.833     0.912


## Virtual Screening Test: Hard-Negative Library

The scaffold-split test set (chemical families unseen in training) was used as a screening library: all 997 measured inactives plus 52 randomly sampled actives, giving 1,049 molecules with 5.0% actives. Sampling was repeated 100 times. Models were trained on the scaffold training set only, and ties in scores were broken at random.

| Metric | Random Forest | Tanimoto 1-NN |
|--------|---------------|----------------|
| ROC-AUC | 0.895 (0.857 to 0.931) | 0.875 (0.833 to 0.912) |
| PR-AUC | 0.505 (0.404 to 0.596) | 0.315 (0.248 to 0.370) |
| EF at 1% (maximum 20.2) | 16.5 (12.8 to 20.2) | 6.7 (3.7 to 9.2) |
| EF at 5% | 9.2 (7.6 to 10.7) | 8.3 (7.2 to 9.5) |
| Precision at 1% (11 molecules) | 0.82 (0.64 to 1.00) | 0.33 (0.18 to 0.46) |
| Precision at 5% (53 molecules) | 0.46 (0.38 to 0.53) | 0.41 (0.36 to 0.47) |

Values are means over 100 resamples, with the 5th to 95th percentile in parentheses. The random baseline is EF 1.0 and PR-AUC 0.05.

Overall ranking quality (ROC-AUC) is close for the two methods, but Random Forest is much better at the top of the ranking: EF at 1% is 16.5, about 82% of the maximum possible value, against 6.7 for the similarity rule. The advantage shrinks at 5% (9.2 vs 8.3). Because 1% of the library is only 11 molecules, early-enrichment values are noisy. The intervals reflect only the random choice of actives, not model or split variability.

In [4]:
from rdkit.Chem.Scaffolds import MurckoScaffold

summary.to_csv(PLOTS.parent / "screening_hard_negatives.csv", index=False)

smiles_test = df['canonical_smiles'].values[test_idx]

# One representative library (fixed seed, same construction as above)
rng = np.random.default_rng(0)
chosen = rng.choice(act_idx, size=n_act, replace=False)
lib = np.concatenate([chosen, inact_idx])
K = int(np.ceil(0.05 * len(lib)))
order = np.argsort(-(rf_scores[lib] + 1e-9 * rng.random(len(lib))))
top = lib[order[:K]]

def diversity_report(idx, name):
    S = tanimoto_matrix(X_test[idx], X_test[idx])
    pairwise = S[np.triu_indices(len(idx), k=1)].mean()
    scaffolds = {MurckoScaffold.MurckoScaffoldSmiles(smiles=smiles_test[i], includeChirality=False) for i in idx}
    novelty = np.median(tanimoto_matrix(X_test[idx], X_train[y_train == 1]).max(axis=1))
    print(f"{name}: n={len(idx)} | unique scaffolds={len(scaffolds)} | "
          f"mean pairwise Tanimoto={pairwise:.2f} | median max similarity to training actives={novelty:.2f}")

print(f"Actives among the top {K}: {int(y_test[top].sum())}")
diversity_report(top, f"Top {K} Random Forest hits")
diversity_report(chosen, f"All {n_act} sampled actives")

Actives among the top 53: 23
Top 53 Random Forest hits: n=53 | unique scaffolds=53 | mean pairwise Tanimoto=0.21 | median max similarity to training actives=0.80
All 52 sampled actives: n=52 | unique scaffolds=52 | mean pairwise Tanimoto=0.18 | median max similarity to training actives=0.75


## Diversity and Novelty of the Top Hits

In one representative library (seed 0, 52 actives and 997 inactives), the top 5% of the Random Forest ranking (53 molecules) contains 23 actives (precision 0.43, within the range of the 100 resamples). Diversity and novelty were measured on these 53 molecules and compared with all 52 sampled actives.

| Set | Mean pairwise Tanimoto | Median max similarity to training actives |
|-----|------------------------|--------------------------------------------|
| Top 53 Random Forest hits | 0.21 | 0.80 |
| All 52 sampled actives | 0.18 | 0.75 |

The top hits are about as diverse as the actives themselves, so the early enrichment is not driven by a single chemical series. They are also close analogs of known training actives (median nearest-neighbour similarity 0.80), so the model recovers analogs of known chemotypes and not novel scaffolds. Counting unique Murcko scaffolds is uninformative here, because the scaffold split places the rarest scaffolds in the test set and almost every test molecule has its own scaffold (53 of 53 and 52 of 52). The top-hit set also contains 30 inactive molecules, which are included in these statistics.

In [5]:

approved = {
    "gefitinib": "COc1cc2ncnc(Nc3ccc(F)c(Cl)c3)c2cc1OCCCN1CCOCC1",
    "erlotinib": "COCCOc1cc2ncnc(Nc3cccc(C#C)c3)c2cc1OCCOC",
    "lapatinib": "CS(=O)(=O)CCNCc1ccc(-c2ccc3ncnc(Nc4ccc(OCc5cccc(F)c5)c(Cl)c4)c3c2)o1",
    "osimertinib": "C=CC(=O)Nc1cc(Nc2nccc(-c3cn(C)c4ccccc34)n2)c(OC)cc1N(C)CCN(C)C",
}

# Using the flat InChIKey (same key used for deduplication)
flat = df['inchikey_flat'].values
found = {}
train_set = set(train_idx.tolist())
for name, smi in approved.items():
    key = Chem.MolToInchiKey(Chem.MolFromSmiles(smi)).split('-')[0]
    hits = np.where(flat == key)[0]
    if len(hits) == 0:
        print(f"{name}: not in the dataset")
    else:
        i = int(hits[0])
        found[name] = i
        print(f"{name}: found | label={y[i]} | split={'train' if i in train_set else 'test'}")

# Retrain Random Forest
drug_idx = set(found.values())
train_nd = np.array([i for i in train_idx if i not in drug_idx])
rf_nd = RandomForestClassifier(n_estimators=500, class_weight='balanced', random_state=42, n_jobs=-1)
rf_nd.fit(X[train_nd], y[train_nd])

# Score the drugs 
inact_scores = rf_nd.predict_proba(X_test[inact_idx])[:, 1]
train_actives = X[train_nd][y[train_nd] == 1]
print()
for name, i in found.items():
    score = rf_nd.predict_proba(X[[i]])[0, 1]
    pct = (inact_scores < score).mean()
    sim = tanimoto_matrix(X[[i]], train_actives).max()
    print(f"{name}: score={score:.2f} | above {pct:.1%} of {len(inact_idx)} inactives | "
          f"max similarity to a training active={sim:.2f}")

gefitinib: found | label=1 | split=train
erlotinib: found | label=1 | split=train
lapatinib: found | label=1 | split=train
osimertinib: found | label=1 | split=train

gefitinib: score=0.90 | above 97.3% of 997 inactives | max similarity to a training active=0.92
erlotinib: score=0.98 | above 99.9% of 997 inactives | max similarity to a training active=0.90
lapatinib: score=1.00 | above 100.0% of 997 inactives | max similarity to a training active=0.87
osimertinib: score=0.97 | above 99.7% of 997 inactives | max similarity to a training active=0.89


## Sanity Check with Approved EGFR Inhibitors

Gefitinib, erlotinib, lapatinib and osimertinib were located in the dataset through their flat InChIKey. All four are labelled active and fell in the training set of the scaffold split. They were removed from the training set, Random Forest was retrained, and each drug was scored against the 997 measured inactives of the scaffold test set.

| Drug | RF score | Scores above | Max similarity to a training active |
|------|----------|--------------|--------------------------------------|
| Gefitinib | 0.90 | 97.3% of inactives | 0.92 |
| Erlotinib | 0.98 | 99.9% | 0.90 |
| Lapatinib | 1.00 | 100.0% | 0.87 |
| Osimertinib | 0.97 | 99.7% | 0.89 |

The model ranks all four approved drugs near the top of the measured inactives, as expected for known inhibitors. This is a plausibility check, not a novelty test: close analogs of each drug remain in the training set (maximum similarity 0.87 to 0.92), so the result shows that the model recognizes known EGFR chemotypes, and it covers only four compounds.

In [6]:
ids_test = df['molecule_chembl_id'].values[test_idx]

shortlist = pd.DataFrame({
    "molecule_chembl_id": ids_test[top],
    "smiles": smiles_test[top],
    "rf_score": rf_scores[top],
    "max_sim_training_active": tanimoto_matrix(X_test[top], X_train[y_train == 1]).max(axis=1),
    "measured_label": y_test[top],
}).sort_values("rf_score", ascending=False)

shortlist.to_csv(PLOTS.parent / "screening_top_hits.csv", index=False)
print(len(shortlist), "top hits saved |", int(shortlist['measured_label'].sum()), "measured actives")
shortlist.head(10)

53 top hits saved | 23 measured actives


,molecule_chembl_id,smiles,rf_score,max_sim_training_active,measured_label
0,CHEMBL5785692,CC(C)(C)N1CCC(N2C=C([C@@H](Nc3cc(Cl)c4ncc(C#N)...,0.996,0.817204,1
1,CHEMBL5911677,CC(C)(C)N1CCC(N2C=C([C@@H](Nc3cc(Cl)c4ncc(C#N)...,0.994,0.781250,1
2,CHEMBL3939199,COc1cc2c(Nc3ccc(Cl)c(Cl)c3)ncnc2cc1OCc1nc(CN2C...,0.988,0.788732,1
3,CHEMBL3921861,COc1cc2c(Nc3ccc(Br)c(Cl)c3)ncnc2cc1O[C@H]1CO[C...,0.986,0.774648,1
4,CHEMBL3905012,COc1cc2c(Nc3ccc(Cl)c(Cl)c3)ncnc2cc1OCCN1CCCC1,0.982,0.981481,1
5,CHEMBL5815497,N#Cc1cnc2c(Cl)cc(N[C@H](C3=CN(Cc4ccccn4)NN3)c3...,0.980,0.797753,1
6,CHEMBL3752337,C=CC(=O)Nc1cccc(-n2c(=O)cc(-c3ccc(-c4ccccc4)cc...,0.976,0.809524,0
7,CHEMBL3916759,COc1cc2c(Nc3ccc(Cl)c(Cl)c3)ncnc2cc1OCC1CN(Cc2c...,0.972,0.837838,1
8,CHEMBL197568,CCOC(=O)c1cn2ncnc(Nc3ccc4c(cnn4Cc4ccccc4)c3)c2...,0.970,0.897059,0
9,CHEMBL1683971,Cc1ncnc(Nc2ccc(OCc3cccc(F)c3)c(Cl)c2)c1C#CCN1C...,0.966,0.727273,0
